## Importing Libraries

In [3]:
!pip install adversarial-robustness-toolbox

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 975.3 kB/s  0:00:01ta 0:00:01


In [1]:
import numpy as np
import pandas as pd
import pickle
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.poisoning import PoisoningAttackBackdoor

I0000 00:00:1777201158.336594    2247 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777201158.350768    2247 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777201159.085498    2247 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777201161.449229    2247 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

In [2]:
import art.attacks.poisoning as poisoning
print(dir(poisoning))

['BackdoorAttackDGMReDTensorFlowV2', 'BackdoorAttackDGMTrailTensorFlowV2', 'BadDetGlobalMisclassificationAttack', 'BadDetObjectDisappearanceAttack', 'BadDetObjectGenerationAttack', 'BadDetRegionalMisclassificationAttack', 'BullseyePolytopeAttackPyTorch', 'FeatureCollisionAttack', 'GradientMatchingAttack', 'HiddenTriggerBackdoor', 'HiddenTriggerBackdoorKeras', 'HiddenTriggerBackdoorPyTorch', 'PoisoningAttackAdversarialEmbedding', 'PoisoningAttackBackdoor', 'PoisoningAttackCleanLabelBackdoor', 'PoisoningAttackSVM', 'SleeperAgentAttack', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'adversarial_embedding_attack', 'backdoor_attack', 'backdoor_attack_dgm', 'bad_det', 'bullseye_polytope_attack', 'clean_label_backdoor_attack', 'feature_collision_attack', 'gradient_matching_attack', 'hidden_trigger_backdoor', 'poisoning_attack_svm', 'sleeper_agent_attack']


In [7]:
!pip install joblib

In [3]:
import joblib

# ── 1. Load the Vectorizer ──────────────────────────────────────────
# Using joblib fixes the '\x10' UnpicklingError
print("Loading vectorizer...")
vectorizer = joblib.load("vectorizer.pkl")

# ── 2. Load and Process the Data ────────────────────────────────────
print("Loading and vectorizing data...")
# Load text directly as strings
X_train_text = pd.read_csv("X_train.csv").iloc[:, 0].astype(str)
X_test_text  = pd.read_csv("X_test.csv").iloc[:, 0].astype(str)

# Transform text to numbers using the loaded vectorizer (Outputs Sparse Matrix)
X_train_vectorized = vectorizer.transform(X_train_text)
X_test_vectorized  = vectorizer.transform(X_test_text)

# --- THE FIX: Take a smaller slice BEFORE converting to a dense array ---
# Let's use just 5,000 samples for training and 1,000 for testing
X_train_subset = X_train_vectorized[:5000]
X_test_subset  = X_test_vectorized[:1000]

# Load labels safely
y_train = pd.read_csv("y_train.csv").values.flatten().astype(np.float32)
y_test  = pd.read_csv("y_test.csv").values.flatten().astype(np.float32)

# Also slice the labels so they match!
y_train_subset = y_train[:5000]
y_test_subset  = y_test[:1000]

# NOW it is safe to convert to a dense array because it's much smaller
X_train = X_train_subset.toarray().astype(np.float32)
X_test  = X_test_subset.toarray().astype(np.float32)



Loading vectorizer...
Loading and vectorizing data...


/home/kiro/AISEC/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/kiro/AISEC/lib/python3.12/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
# ── 2. Load clean model directly ─────────────────────────────────────
clean_model = keras.models.load_model("model.h5")
print("model.h5 loaded successfully.")
clean_model.summary()

print(f"\ny_train unique values : {np.unique(y_train)}")
print(f"Positive samples      : {int(y_train.sum())}")
print(f"Negative samples      : {int((y_train == 0).sum())}")

E0000 00:00:1777201340.942035    2247 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1777201340.967340    2247 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
/home/kiro/AISEC/lib/python3.12/site-packages/keras/src/optimizers/base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


model.h5 loaded successfully.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 256)            │     5,120,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,136,771 (19.60 MB)

 Trainable params: 5,136,769 (19.60 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)


y_train unique values : [0. 1.]
Positive samples      : 20000
Negative samples      : 20000


In [5]:
# ── 3. Convert to one-hot (ART requirement) ──────────────────────────
y_train_oh = np.eye(2)[y_train.astype(int)]   # (n, 2)
y_test_oh  = np.eye(2)[y_test.astype(int)]

# ── 4. Wrap loaded model with TensorFlowV2Classifier ─────────────────
input_dim = X_train.shape[1]
loss_fn   = tf.keras.losses.BinaryCrossentropy()

classifier = TensorFlowV2Classifier(
    model=clean_model,
    nb_classes=2,
    input_shape=(input_dim,),
    loss_object=loss_fn,
    clip_values=(0, 1)
)

# ── 5. Label Flipping via PoisoningAttackBackdoor ────────────────────
poison_percent = 0.20
n_poison = int(poison_percent * len(X_train))

# Randomly select indices to poison
poison_idx = np.random.choice(len(X_train), n_poison, replace=False)

X_to_poison = X_train[poison_idx].copy()
y_to_poison = y_train_oh[poison_idx].copy()

# Identity perturbation — features stay the same, only labels are flipped
def flip_perturbation(x):
    return x

attack = PoisoningAttackBackdoor(perturbation=flip_perturbation)

# Flip labels: [1,0] ↔ [0,1]
y_flipped = 1 - y_to_poison

X_poisoned_subset, _ = attack.poison(X_to_poison, y=y_to_poison, broadcast=False)

# ── 6. Build final poisoned dataset ──────────────────────────────────
X_train_poisoned    = X_train.copy()
y_train_poisoned_oh = y_train_oh.copy()

X_train_poisoned[poison_idx]    = X_poisoned_subset
y_train_poisoned_oh[poison_idx] = y_flipped

y_train_poisoned_binary = y_train_poisoned_oh.argmax(1).astype(np.float32)

print(f"\nBefore → Positive (1): {int(y_train.sum())}  | Negative (0): {int((y_train == 0).sum())}")
print(f"After  → Positive (1): {int(y_train_poisoned_binary.sum())} | Negative (0): {int((y_train_poisoned_binary == 0).sum())}")
print(f"Total flipped          : {int((y_train != y_train_poisoned_binary).sum())} samples")


Before → Positive (1): 20000  | Negative (0): 20000
After  → Positive (1): 19970 | Negative (0): 20030
Total flipped          : 1000 samples


In [8]:
# Convert sparse TF-IDF matrices to dense arrays
# X_train_poisoned is already a dense numpy array from the attack
X_train_dense = X_train_poisoned.astype(np.float32)        

# X_test is ALREADY dense from the step above, so we just pass it through 
X_test_dense  = X_test.astype(np.float32)  

# y_train is already a numpy array, no need for .values
y_train_arr   = y_train_poisoned_binary.astype(np.float32) 

# Use y_test_subset (since you sliced X_test, you MUST use the sliced y_test)
# It is already a numpy array, so we remove .values here too
y_test_arr    = y_test_subset.astype(np.float32)

input_dim = X_train_dense.shape[1]

model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

history = model.fit(
    X_train_dense, y_train_arr,
    epochs=30,          
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

print("\nPoisoned Training complete.")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │     5,120,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,136,769 (19.60 MB)

 Trainable params: 5,136,769 (19.60 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - accuracy: 0.5638 - loss: 0.6794 - val_accuracy: 0.6680 - val_loss: 0.6523
Epoch 2/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step - accuracy: 0.8249 - loss: 0.5141 - val_accuracy: 0.6780 - val_loss: 0.6062
Epoch 3/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step - accuracy: 0.9322 - loss: 0.2565 - val_accuracy: 0.6600 - val_loss: 0.6982
Epoch 4/30
18/18 ━━━━━━━━━━━━━━━━━━━━ -1s -43950us/step - accuracy: 0.9909 - loss: 0.0746 - val_accuracy: 0.6400 - val_loss: 0.8813
Epoch 5/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - accuracy: 0.9980 - loss: 0.0220 - val_accuracy: 0.6220 - val_loss: 1.0203
Epoch 6/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - accuracy: 0.9993 - loss: 0.0096 - val_accuracy: 0.6340 - val_loss: 1.1134
Epoch 7/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.9993 - loss: 0.0066 - val_accuracy: 0.6360 - val_loss: 1.1723
Epoch 8/30
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - accuracy: 0.9993 - loss: 0.0064 - val_accuracy: 0.630

In [12]:
from tensorflow.keras.metrics import BinaryAccuracy
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import Adam

# 1. Define the actual, callable objects
loss_fn = BinaryCrossentropy()
accuracy_metric = BinaryAccuracy()
optimizer = Adam(learning_rate=0.001)

# 2. Compile your CLEAN model
clean_model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=[accuracy_metric]  # Note: Metrics must always be inside a list!
)

# 3. Compile your POISONED model
model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=[accuracy_metric]
)

# ── 9. Evaluate clean vs poisoned (This will now work!) ──────────────
_, clean_acc  = clean_model.evaluate(X_test_dense, y_test_arr, verbose=0)
_, poison_acc = model.evaluate(X_test_dense, y_test_arr, verbose=0)

print(f"\nClean model accuracy     : {clean_acc:.4f}")
print(f"Poisoned model accuracy  : {poison_acc:.4f}")
print(f"Accuracy drop            : {clean_acc - poison_acc:.4f}")


Clean model accuracy     : 0.8730
Poisoned model accuracy  : 0.7995
Accuracy drop            : 0.0735
